<a href="https://colab.research.google.com/github/YOUR_GITHUB_USER/YOUR_REPO/blob/main/track1_emerging_tech_fixit_agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🛠️ Track 1 · Emerging Technology
## FixIt: A Multimodal AI Agent for Facilities Maintenance

**Coaching session · ~60 min**

### The real-world problem
Malls, campuses, hospitals and town councils get hundreds of fault reports a week, usually as a WhatsApp photo plus a vague message: *"aircon dripping near lift lobby level 3"*. A human dispatcher then has to:
1. work out what's broken and how urgent it is,
2. look up the equipment and its repair history,
3. check that spare parts are in stock,
4. find a technician with the right skills, and
5. raise a work order.

It's slow, inconsistent, and nights and weekends are the worst.

### The emerging-tech angle
Two capabilities that only became practical recently:
- **Multimodal models** can *see* the photo and read the message together.
- **AI agents** (LLMs with **function calling**) can decide which of your company's systems to call, in what order, and act on the results.

### What you will build
| Step | Technique | What it does |
|---|---|---|
| Triage | Multimodal + **structured output** | Photo + text ➜ typed JSON (category, severity, hazard, parts) |
| Dispatch | **Function calling / agent** | The model calls your tools to look up assets, history, parts and staff, then raises a work order |
| Guardrail | **Human-in-the-loop** | Critical jobs wait for a supervisor instead of auto-dispatching |
| Demo | **Gradio** | Upload a photo, get a dispatched job |

## Step 1: Install Dependencies
- `google-genai`: Google's Gen AI SDK for Gemini (already preinstalled in Colab).
- `gradio`: the web UI for the demo.

`pydantic` and `Pillow` (images) are already in Colab.

The first line records the versions of a few core packages Colab depends on, so the install below doesn't upgrade them and break Colab. If Colab asks you to **restart the session** after installing, click restart and continue from Step 2.

In [ ]:
# Keep Colab's own versions of these packages to avoid dependency conflicts
!pip freeze | grep -iE '^(google-auth|opentelemetry-(api|sdk))==' > /tmp/colab-constraints.txt
!pip install -q -c /tmp/colab-constraints.txt google-genai gradio==6.29.1

## Step 2: Setup Google Gemini API Key
1. Go to [Google AI Studio](https://aistudio.google.com/).
2. Sign in with your Google account and click **Get API key**.
3. In Colab, click the 🔑 **Secrets** icon on the left sidebar.
4. Create a new secret named `GEMINI_API_KEY` and paste your key.
5. Turn on the **Notebook access** toggle for that secret.

We also define `ask_gemini()`, a small wrapper that waits and retries when the free tier says *"429: too many requests"*. Free-tier keys only allow a few requests per minute, so every team will hit this during the hackathon.

In [ ]:
import time
from google import genai
from google.genai import types
from google.colab import userdata

# Securely load the API key from Colab Secrets
try:
    GOOGLE_API_KEY = userdata.get('GEMINI_API_KEY')
    client = genai.Client(api_key=GOOGLE_API_KEY)
    print("✅ Gemini API Key configured successfully!")
except Exception as e:
    print("❌ Error: Please ensure you added 'GEMINI_API_KEY' to your Colab Secrets and turned on 'Notebook access'.")
    print("   Details:", e)
    raise

# One place to change the model for the whole notebook
MODEL = "gemini-2.5-flash"

def ask_gemini(contents, config=None, retries=3):
    """Call Gemini, waiting and retrying if we hit the free-tier rate limit."""
    for attempt in range(retries):
        try:
            return client.models.generate_content(model=MODEL, contents=contents, config=config)
        except Exception as e:
            if "429" in str(e) and attempt < retries - 1:
                wait = 20 * (attempt + 1)
                print(f"⏳ Rate limited by the free tier, retrying in {wait}s...")
                time.sleep(wait)
            else:
                raise

## Step 3: The Company's Back-Office Systems (Mock Data)
A real agent would call your maintenance system's API (a *CMMS*), your inventory database and your HR roster. Here we fake them with Python dictionaries for a fictional building, **Marina Tech Park**.

> 💡 **Hackathon tip:** judges don't expect real integrations. A clean mock that *looks like* a real API is enough to prove the idea.

In [ ]:
ASSETS = {
    "AHU-L3-02": {"type": "Air Handling Unit", "location": "Level 3, lift lobby B", "installed": 2016, "skill": "HVAC"},
    "FCU-L5-11": {"type": "Fan Coil Unit", "location": "Level 5, meeting room 5.11", "installed": 2019, "skill": "HVAC"},
    "PUMP-B1-01": {"type": "Water Pump", "location": "Basement 1, pump room", "installed": 2014, "skill": "Plumbing"},
    "DB-L2-04": {"type": "Electrical Distribution Board", "location": "Level 2, riser cupboard", "installed": 2015, "skill": "Electrical"},
    "LIFT-B": {"type": "Passenger Lift", "location": "Lift lobby B, all levels", "installed": 2012, "skill": "Lift"},
    "TOILET-L3-M": {"type": "Washroom (male)", "location": "Level 3, near lift lobby B", "installed": 2016, "skill": "Plumbing"},
}

HISTORY = {
    "AHU-L3-02": [
        {"date": "2026-06-02", "issue": "Condensate drain pipe clogged, water leaking", "fix": "Cleared drain line"},
        {"date": "2026-08-19", "issue": "Water dripping from ceiling below unit", "fix": "Cleared drain line again"},
    ],
    "PUMP-B1-01": [{"date": "2026-03-11", "issue": "Bearing noise", "fix": "Replaced bearing"}],
    "LIFT-B": [{"date": "2026-09-30", "issue": "Door sensor fault", "fix": "Sensor recalibrated"}],
}

SPARE_PARTS = {
    "condensate drain pump": 0,
    "drain pipe": 12,
    "air filter": 40,
    "circuit breaker": 6,
    "pump bearing": 2,
    "door sensor": 1,
    "tap cartridge": 8,
    "fluorescent tube": 30,
}

TECHNICIANS = [
    {"id": "T01", "name": "Ravi", "skills": ["HVAC", "Plumbing"], "on_duty": True},
    {"id": "T02", "name": "Mei Ling", "skills": ["Electrical"], "on_duty": True},
    {"id": "T03", "name": "Hafiz", "skills": ["Lift", "Electrical"], "on_duty": False},
    {"id": "T04", "name": "Jun Wei", "skills": ["HVAC"], "on_duty": True},
]

WORK_ORDERS = []
print(f"Loaded {len(ASSETS)} assets, {len(SPARE_PARTS)} spare parts, {len(TECHNICIANS)} technicians.")

## Step 4: Multimodal Triage with Structured Output
Free-text answers are hard for code to use. Instead we give Gemini a **Pydantic schema** and it must answer with JSON that fits it. The SDK then hands us a real Python object (`response.parsed`), so there's no regex parsing.

`Literal[...]` limits a field to a fixed list of values, so severity is always one of four words your code can rely on.

In [ ]:
from typing import Literal
from pydantic import BaseModel, Field

class Triage(BaseModel):
    category: Literal["HVAC", "Plumbing", "Electrical", "Lift", "Structural", "Cleaning", "Other"]
    issue_summary: str = Field(description="One sentence describing the fault, for a technician")
    severity: Literal["low", "medium", "high", "critical"]
    safety_hazard: bool = Field(description="True if people could be hurt (electric shock, slip, fire, trapped in lift)")
    likely_parts: list[str] = Field(description="Spare parts probably needed, in plain words, e.g. 'drain pipe'")
    location_hint: str = Field(description="Where the fault is, copied from the report")

TRIAGE_PROMPT = """You are a facilities maintenance triage expert.
Read the fault report (and the photo, if there is one) and classify the problem.
Severity guide:
- critical: immediate danger to people, or a lift with someone trapped
- high: service down for many people, or damage spreading (e.g. active leak)
- medium: something broken but contained
- low: cosmetic or minor inconvenience

Fault report: """

def triage(report_text, image=None):
    contents = [TRIAGE_PROMPT + report_text]
    if image is not None:
        contents.append(image)  # the SDK accepts PIL images directly
    response = ask_gemini(
        contents,
        types.GenerateContentConfig(response_mime_type="application/json", response_schema=Triage),
    )
    return response.parsed

sample_report = "Water keeps dripping from the ceiling next to the lift lobby on level 3, there's a puddle on the floor and people are walking around it."
result = triage(sample_report)
print(result.model_dump_json(indent=2))

### 📸 Try it with your own photo (optional)
Take a photo of anything "broken" around you (a messy desk, a flickering light, a dripping tap) and upload it. Notice how the model combines what it **sees** with what you **wrote**.

In [ ]:
from PIL import Image
from google.colab import files

uploaded = files.upload()  # pick a .jpg or .png
if uploaded:
    photo = Image.open(next(iter(uploaded)))
    photo.thumbnail((1024, 1024))  # smaller image = faster and cheaper
    display(photo)
    print(triage("See photo. Found this on level 5 near meeting room 5.11.", image=photo).model_dump_json(indent=2))

## Step 5: Give the Agent Tools (Function Calling)
This is what turns a chatbot into an **agent**. We write normal Python functions and pass them to Gemini. The SDK reads each function's **name, type hints and docstring** and describes them to the model.

The model then decides on its own *which* tool to call and *with what arguments*. The SDK runs the function and sends the result back, and this loops until the model has its final answer. This is called **automatic function calling**.

> ✍️ Docstrings matter: they are the model's only documentation for your tools. Vague docstrings lead to wrong tool calls.

We also log every call in `TOOL_LOG` so we can show the agent's "thinking trail" in the demo.

In [ ]:
TOOL_LOG = []

def _log(tool, args, result):
    TOOL_LOG.append({"tool": tool, "args": args, "result": result})
    return result

def find_asset(location_keywords: str) -> dict:
    """Find building equipment near a location.

    Args:
        location_keywords: words describing where the fault is, e.g. "level 3 lift lobby".
    Returns:
        Matching assets with their asset_id, type, location and the skill needed to fix them.
    """
    words = [w for w in location_keywords.lower().replace(",", " ").split() if len(w) > 1 or w.isdigit()]
    matches = {
        asset_id: info for asset_id, info in ASSETS.items()
        if sum(w in info["location"].lower() for w in words) >= 2
    }
    return _log("find_asset", {"location_keywords": location_keywords}, {"matches": matches})

def get_maintenance_history(asset_id: str) -> dict:
    """Get past repairs for an asset. Repeated faults suggest the root cause was never fixed.

    Args:
        asset_id: the asset ID returned by find_asset, e.g. "AHU-L3-02".
    """
    return _log("get_maintenance_history", {"asset_id": asset_id}, {"history": HISTORY.get(asset_id, [])})

def check_spare_parts(part_name: str) -> dict:
    """Check how many of a spare part are in the store room.

    Args:
        part_name: plain-language part name, e.g. "drain pipe".
    """
    hits = {p: qty for p, qty in SPARE_PARTS.items() if part_name.lower() in p or p in part_name.lower()}
    return _log("check_spare_parts", {"part_name": part_name}, {"stock": hits or "part not stocked"})

def get_on_duty_technician(skill: str) -> dict:
    """Find technicians who are on duty now and have the needed skill.

    Args:
        skill: one of HVAC, Plumbing, Electrical, Lift.
    """
    techs = [t for t in TECHNICIANS if skill in t["skills"] and t["on_duty"]]
    return _log("get_on_duty_technician", {"skill": skill}, {"technicians": techs})

def create_work_order(asset_id: str, issue_summary: str, severity: str,
                      technician_id: str, parts_needed: list[str]) -> dict:
    """Raise a work order to dispatch a technician. Call this last, once you have all the details.

    Args:
        asset_id: asset to repair.
        issue_summary: one-sentence description of the fault.
        severity: low, medium, high or critical.
        technician_id: ID from get_on_duty_technician, or "UNASSIGNED" if nobody is available.
        parts_needed: spare parts the technician should bring.
    """
    # Guardrail: critical jobs need a human to approve them
    status = "PENDING_SUPERVISOR_APPROVAL" if severity == "critical" else "DISPATCHED"
    order = {
        "work_order_id": f"WO-{1001 + len(WORK_ORDERS)}",
        "asset_id": asset_id, "issue_summary": issue_summary, "severity": severity,
        "technician_id": technician_id, "parts_needed": parts_needed, "status": status,
    }
    WORK_ORDERS.append(order)
    return _log("create_work_order", {"asset_id": asset_id, "severity": severity}, order)

TOOLS = [find_asset, get_maintenance_history, check_spare_parts, get_on_duty_technician, create_work_order]
print("🧰 Tools ready:", [t.__name__ for t in TOOLS])

## Step 6: Run the Agent
The **system instruction** is the agent's job description. Keep it short, give a clear order of work, and say what *not* to do (e.g. never invent IDs).

In [ ]:
AGENT_INSTRUCTIONS = """You are FixIt, the dispatcher for Marina Tech Park.
You receive a triaged fault report. Work through these steps using your tools:
1. Find the asset at the reported location.
2. Check its maintenance history. If the same problem happened before, say so and recommend a root-cause fix.
3. Check stock for each likely part.
4. Find an on-duty technician with the right skill.
5. Create the work order.
Rules: only use IDs returned by the tools, never invent them. If a part is out of stock, still dispatch but flag it.
Finish with a short summary for the supervisor (max 120 words): work order ID, status, technician, parts, and any warning."""

def run_agent(triage_result):
    TOOL_LOG.clear()
    config = types.GenerateContentConfig(system_instruction=AGENT_INSTRUCTIONS, tools=TOOLS)
    response = ask_gemini(f"Triaged fault report:\n{triage_result.model_dump_json(indent=2)}", config)
    return response.text

summary = run_agent(result)

print("🔧 Agent's tool calls:")
for i, call in enumerate(TOOL_LOG, 1):
    print(f"  {i}. {call['tool']}({call['args']})")
print("\n📋 Summary for supervisor:\n", summary)

### 🤔 Coaching discussion (5 min)
- Look at the tool-call trail. Did the agent notice that **AHU-L3-02 has leaked twice before**? That's an insight a rushed human dispatcher often misses.
- `condensate drain pump` has **0 in stock**. Did the agent flag it?
- Change the report to *"someone is stuck in lift B and the alarm isn't working"*. Does the severity become `critical`, and does the work order wait for approval?
- What would go wrong if `create_work_order` **sent a real SMS** and the model called it twice? (Hint: *idempotency*, and why humans approve critical steps.)

## Step 7: Build the Demo UI with Gradio 🏆
Photo and message in; triage, agent trail and work order out. Showing the **tool-call trail** in the UI lets judges *see* the agent working, which is much more convincing than a single answer.

In [ ]:
import gradio as gr

def fixit_pipeline(photo, report):
    if not report and photo is None:
        return {}, "Please upload a photo or describe the fault.", []
    try:
        t = triage(report or "See photo.", image=photo)
        summary = run_agent(t)
    except Exception as e:
        return {}, f"⚠️ Sorry, something went wrong: {e}", []
    trail = [[c["tool"], str(c["args"]), str(c["result"])[:150]] for c in TOOL_LOG]
    return t.model_dump(), summary, trail

with gr.Blocks(title="FixIt Agent") as demo:
    gr.Markdown("# 🛠️ FixIt: AI Maintenance Dispatcher\nSnap a photo of a fault, describe it, and let the agent triage and dispatch.")
    with gr.Row():
        with gr.Column():
            photo_in = gr.Image(type="pil", label="Photo (optional)")
            report_in = gr.Textbox(label="Fault report", lines=3,
                                   placeholder="e.g. Water dripping from ceiling at level 3 lift lobby")
            go = gr.Button("Triage & dispatch", variant="primary")
            gr.Examples(
                examples=[
                    [None, "Water keeps dripping from the ceiling next to the lift lobby on level 3."],
                    [None, "Burning smell and sparks from the electrical cupboard on level 2!"],
                    [None, "Toilet tap on level 3 near lift lobby B won't stop running."],
                ],
                inputs=[photo_in, report_in],
            )
        with gr.Column():
            triage_out = gr.JSON(label="1️⃣ Triage")
            summary_out = gr.Markdown(label="3️⃣ Supervisor summary")
    trail_out = gr.Dataframe(headers=["Tool", "Arguments", "Result"], label="2️⃣ Agent tool-call trail", wrap=True)
    go.click(fixit_pipeline, inputs=[photo_in, report_in], outputs=[triage_out, summary_out, trail_out])

# Launch it! (This will generate a public URL you can share with judges)
demo.launch(share=True, theme="soft")

## 🚀 Make It Your Own: Hackathon Extensions
Pick one or two. Don't try all of them.
- **Voice reports:** Gemini accepts audio, so cleaners can just send a voice note in any language.
- **Real channel:** connect to a Telegram bot (`python-telegram-bot`) so reports come from a phone, not a web page.
- **Predictive maintenance:** add a tool that reads (mock) IoT sensor data, such as vibration or temperature, and raises a job *before* things break.
- **Approval flow:** add an "Approve" button that changes `PENDING_SUPERVISOR_APPROVAL` to `DISPATCHED`.
- **Different domain, same pattern:** clinic triage, IT helpdesk, insurance claims (photo of car damage), farm pest detection.

## 🎤 Pitching to Judges
- Lead with a **number**: *"Dispatch time goes from 25 minutes to 30 seconds, and repeat faults get spotted automatically."*
- Show the **tool-call trail**. It proves this is an agent, not just a chatbot.
- Show the **guardrail**. Judges ask *"what if the AI is wrong?"*, so have the answer on screen.

## 🛡️ Responsible AI Checklist
- [ ] Critical actions need a human to approve them.
- [ ] Tools validate their inputs. Never trust IDs the model makes up.
- [ ] Photos may contain faces or personal data, so blur or don't store them.
- [ ] Log every tool call for auditing (you already do, in `TOOL_LOG`).